# Use spline to get training angles

In [3]:
import numpy as np
from scipy.interpolate import CubicSpline
import pandas as pd

# ============================================================
# 1. KNOWN DATA — your 7 known throuples, ordered by increasing t
#    each is (x, y1, y2)
# ============================================================

throuples = [
    (-40, -45, 38),
    (-35, -38, 20),
    (-20, -38, 25),
    (0,   -30, 60),
    (20,  -38, 25),
    (35,  -38, 20),
    (40,  -45, 38),
]

# t position along the line for each of the 7 points above (0=left, 1=right)
t_angles = np.array([0, 0.166, 0.333, 0.5, 0.666, 0.834, 1])

# 7 known pixel_x values, ordered left -> right
pixel_x_vals = np.array([462, 406, 344, 276, 216, 150, 90])
t_pixels = t_angles  # same t values, reused

# ============================================================
# 2. FIT ANGLE SPLINES (cubic, exact through all 7 points)
# ============================================================

data = np.array(throuples)  # shape (7, 3)

spline_x  = CubicSpline(t_angles, data[:, 0])
spline_y1 = CubicSpline(t_angles, data[:, 1])
spline_y2 = CubicSpline(t_angles, data[:, 2])

# ============================================================
# 3. FIT PIXEL_X MAPPING
#    With only 3 known pixel_x values, a projective (Mobius)
#    transform was used to extrapolate. Now that we have all 7
#    measured pixel_x values (matching every t), just spline
#    through them directly -- exact fit, no extrapolation needed.
# ============================================================

spline_px = CubicSpline(t_pixels, pixel_x_vals)

def pixel_x_fn(t):
    return spline_px(t)

# sanity check extremes
print("pixel_x at t=0:", pixel_x_fn(0.0))
print("pixel_x at t=1:", pixel_x_fn(1.0))

# ============================================================
# 4. GENERATE TRAINING DATA
# ============================================================

t_train = np.linspace(0, 1, 200)

df = pd.DataFrame({
    'px': pixel_x_fn(t_train),
    'x':  spline_x(t_train),
    'y1': spline_y1(t_train),
    'y2': spline_y2(t_train),
})

# ============================================================
# 5. ROUND TO DISCRETE VALUES + DROP DUPLICATES
# ============================================================

step = 1  # rounding step, e.g. 1 for whole degrees/pixels, 0.5 for half-steps

df = (df / step).round(0) * step

if float(step).is_integer():
    df = df.astype(int)
# else keep as float (for fractional steps like 0.5)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"{before} rows -> {len(df)} unique after rounding")

# ============================================================
# 6. SAVE
# ============================================================

df.to_csv('training_data.csv', index=False)
print(df.head())
print(df.tail())

pixel_x at t=0: 462.0
pixel_x at t=1: 90.0
200 rows -> 200 unique after rounding
    px   x  y1  y2
0  462 -40 -45  38
1  460 -40 -44  38
2  459 -40 -44  37
3  457 -40 -43  37
4  455 -40 -43  36
     px   x  y1  y2
195  96  40 -43  36
196  95  40 -43  37
197  93  40 -44  37
198  92  40 -44  38
199  90  40 -45  38
